# DeepBio-Scan: Phase 2 - Neural Embedding Pipeline (500k Sequence Atlas)

**Personas Active:**
- `@Embedder-ML` (Model Logic & Inference)
- `@Data-Ops` (Data Pipeline & Parquet Export)

**Hardware Target:** Google Colab T4 (Standard) or A100 (High-RAM)
**Prerequisites:** 
1. Create a `shards` folder in the Colab file browser.
2. Manually upload your `shards_local/*.parquet` files into that `shards` folder.
3. Run the notebook to process them into `vectors/`.
4. Download the `vectors/` folder when finished.

In [ ]:
# @Data-Ops: Dependency Setup
!pip uninstall -y torch_xla
!pip install --upgrade transformers==4.40.2 pandas pyarrow duckdb lancedb accelerate biopython tqdm

In [ ]:
# @Data-Ops: Local Environment Setup (Manual Upload Mode)
import os
import glob
import gc
import shutil
import time
import torch
import pandas as pd
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
from tqdm.auto import tqdm

# 1. Path Configuration (Local /content paths)
ROOT_DIR = "/content"
INPUT_SHARD_DIR = os.path.join(ROOT_DIR, "shards")
OUTPUT_VECTOR_DIR = os.path.join(ROOT_DIR, "vectors")

os.makedirs(INPUT_SHARD_DIR, exist_ok=True)
os.makedirs(OUTPUT_VECTOR_DIR, exist_ok=True)

print(f"📂 Input Directory: {INPUT_SHARD_DIR}")
print(f"📂 Output Directory: {OUTPUT_VECTOR_DIR}")
print("👉 ACTION REQUIRED: Please upload your 'shard_*.parquet' files to the 'shards' folder in the file browser on the left.")

def get_pending_shards():
    """Lists input shards that lack a corresponding output vector file."""
    all_shards = sorted(glob.glob(os.path.join(INPUT_SHARD_DIR, "*.parquet")))
    
    if not all_shards:
        print(f"⚠️ No parquet files found in {INPUT_SHARD_DIR}")
        print("   Please use the file browser (folder icon) to upload your shards.")
        return []

    pending = []
    
    for s_path in all_shards:
        fname = os.path.basename(s_path)
        out_name = f"vector_{fname}"
        out_path = os.path.join(OUTPUT_VECTOR_DIR, out_name)
        
        if not os.path.exists(out_path):
            pending.append(s_path)
        else:
            print(f"⏩ Skipping processed shard: {fname}")
            
    print(f"Found {len(pending)} pending shards.")
    return pending

# 2. Hardware Optimization
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Running on: {device.upper()}")

if device == "cuda":
    # Enable TF32 for faster throughput on Ampere+ (A100/T4)
    torch.backends.cuda.matmul.allow_tf32 = True
    print("🚀 TF32 Matrix Multiplication: ENABLED")

In [ ]:
# @Embedder-ML: Neural Core Initialization
from transformers import AutoTokenizer, AutoModelForMaskedLM, AutoConfig

class NeuralCore:
    def __init__(self, model_name="InstaDeepAI/nucleotide-transformer-v2-50m-multi-species"):
        self.device = device
        print(f"Initializing {model_name}...")
        
        # Monkey-Patch Config for 2TB Roadmap Compliance
        config = AutoConfig.from_pretrained(model_name, trust_remote_code=True)
        config.intermediate_size = 4096 
        
        self.tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
        self.model = AutoModelForMaskedLM.from_pretrained(
            model_name,
            config=config,
            trust_remote_code=True,
            ignore_mismatched_sizes=True
        ).to(self.device)
        self.model.eval()
        
    def l2_normalize(self, vectors: np.ndarray, eps: float = 1e-12) -> np.ndarray:
        norms = np.linalg.norm(vectors, axis=1, keepdims=True)
        norms = np.clip(norms, eps, None)
        return (vectors / norms).astype(np.float32)

    def embed_batch(self, sequences, batch_size=128):
        """Generates 768-d embeddings for a list of sequences."""
        all_vecs = []
        
        for i in tqdm(range(0, len(sequences), batch_size), desc="  Embedding", leave=False):
            batch = sequences[i:i + batch_size]
            
            inputs = self.tokenizer(
                batch,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=1000
            ).to(self.device)
            
            with torch.no_grad():
                outputs = self.model(**inputs, output_hidden_states=True)
                last_hidden_state = outputs.hidden_states[-1]
                attention_mask = inputs["attention_mask"].unsqueeze(-1).expand(last_hidden_state.size()).float()
                
                sum_embeddings = torch.sum(last_hidden_state * attention_mask, 1)
                sum_mask = torch.clamp(attention_mask.sum(1), min=1e-9)
                mean_embeddings = sum_embeddings / sum_mask # (B, Dim)
                
                # Expand 512 -> 768 dim (Zero Padding for LanceDB compatibility)
                current_dim = mean_embeddings.shape[1]
                if current_dim < 768:
                    padding = torch.zeros((mean_embeddings.shape[0], 768 - current_dim), device=self.device)
                    mean_embeddings = torch.cat([mean_embeddings, padding], dim=1)
                elif current_dim > 768:
                    mean_embeddings = mean_embeddings[:, :768]
                
                # Move to CPU immediately
                vecs = mean_embeddings.detach().cpu().numpy().astype(np.float32)
                vecs = self.l2_normalize(vecs)
                all_vecs.append(vecs)
            
            del inputs, outputs, last_hidden_state, mean_embeddings, padding, vecs
            
        return np.concatenate(all_vecs, axis=0)

# Instantiate Core
if 'core' in globals():
    del core
    gc.collect()
    torch.cuda.empty_cache()

core = NeuralCore()

In [ ]:
# @Data-Ops: Shard Processing Loop (The Anti-Crash Guard)

def process_shard(shard_path):
    fname = os.path.basename(shard_path)
    out_name = f"vector_{fname}"
    out_path = os.path.join(OUTPUT_VECTOR_DIR, out_name)
    
    print(f"\nProcessing Shard: {fname}...")
    
    # A. Load
    try:
        df = pd.read_parquet(shard_path)
    except Exception as e:
        print(f"❌ Failed to read {shard_path}: {e}")
        return

    # B. Clean
    # Ensure uppercase and replace 'N' with 'A' (Adenine fallback)
    sequences = (
        df["Sequence"]
        .astype(str)
        .str.upper()
        .str.replace("N", "A")
        .tolist()
    )
    
    # C. Embed
    print(f"  Embedding {len(sequences)} sequences...")
    try:
        vectors = core.embed_batch(sequences, batch_size=128)
    except RuntimeError as e:
        if "out of memory" in str(e):
             print("⚠️ OOM! Retrying with batch_size=64...")
             torch.cuda.empty_cache()
             vectors = core.embed_batch(sequences, batch_size=64)
        else:
             raise e

    # D. Integrate & Export
    df["vector"] = list(vectors)
    
    # Save enriched shard
    print(f"  Saving to Local Storage: {out_name}")
    table = pa.Table.from_pandas(df)
    pq.write_table(table, out_path)
    
    # E. Memory Management (Critical)
    del df, sequences, vectors, table
    gc.collect()
    torch.cuda.empty_cache()
    
    print(f"✅ [SUCCESS] {fname} Processed. Ready for download.")

# --- Execution ---
pending_shards = get_pending_shards()

if not pending_shards:
    print("No pending shards found. Please upload .parquet files to:")
    print(f"  {INPUT_SHARD_DIR}")
else:
    for shard_path in tqdm(pending_shards, desc="Total Progress"):
        process_shard(shard_path)
        
    print("\n🎉 MISSION COMPLETE: All shards processed.")
    print(f"Output files are available in: {OUTPUT_VECTOR_DIR}")